In [ ]:
# ==============================================================================
# PART 1: LOAD DOCUMENTS & EXECUTE RERANKING MODEL
# ==============================================================================

# 1. Install Pinecone libraries
!pip install -q -U pinecone==6.0.1 pinecone-notebooks

# 2. Authenticate with Pinecone
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# 3. Instantiate the Pinecone client
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

# 4. Define your query & documents (mixing Apple fruits vs Apple company)
query = "Tell me about Apple's products"
documents = [
    "The Honeycrisp apple is a crisp, sweet fruit ideal for eating raw or baking pie.",
    "Apple recently announced its new line of iPhones, MacBooks, and Apple Watches at the keynote.",
    "Granny Smith apples are famous for their tart flavor and bright green skin appearance.",
    "The tech giant Apple is rumored to be working on advanced mixed-reality hardware applications.",
    "Organic apples are a great natural source of fiber, vitamins, and antioxidants for a healthy diet."
]

# 5. Call the reranker
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

# 6. Inspect reranked results
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    for i, m in enumerate(matches):
        print(f"Rank {i+1}:")
        print(f" Score: {m.score:.4f}")
        print(f" Text: {m.document.text}\n")

print("--- PART 1 RESULTS ---")
show_reranked_results(query, reranked.data)


# ==============================================================================
# PART 2: SETUP A SERVERLESS INDEX FOR MEDICAL NOTES
# ==============================================================================

# 1. Install data & model libraries
!pip install -q pandas torch transformers

# 2. Import modules & define environment settings
import time
import pandas as pd
from pinecone import ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

cloud = os.getenv('PINECONE_CLOUD', 'aws') 
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

# 3. Create or recreate the index
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,  # Dimensionality of sentence-transformers/all-MiniLM-L6-v2
    metric='cosine',
    spec=spec
)


# ==============================================================================
# PART 3: LOAD THE SAMPLE DATA
# ==============================================================================

# 1. Download & read JSONL
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

# 2. Preview the DataFrame
print("\n--- PART 3 DATA PREVIEW ---")
print("Data shape:", df.shape)
display(df.head())


# ==============================================================================
# PART 4: UPSERT DATA INTO THE INDEX
# ==============================================================================

# 1. Instantiate index client & upsert
index = pc.Index(name=index_name)
index.upsert_from_dataframe(df)

# 2. Wait for availability
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Current vector count: {vector_count}")
    return vector_count > 0

print("\n--- PART 4 UPSERT STATUS ---")
while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
display(index.describe_index_stats())


# ==============================================================================
# PART 5: QUERY & EMBEDDING FUNCTION
# ==============================================================================

# 1. Define your embedding function
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        model_output = model(**encoded_input)
        # Mean pooling across the sequence length (dim=1)
        embedding = model_output.last_hidden_state[0].mean(dim=1)
    return embedding

# 2. Run a semantic search query
question = "patient has severe chest pain and short breath" 
query_vector = get_embedding(question).tolist()

results = index.query(vector=[query_vector], top_k=5, include_metadata=True)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)


# ==============================================================================
# PART 6: DISPLAY & RERANK CLINICAL NOTES
# ==============================================================================

# 1. Display initial search results
def show_results(question, matches):
    print(f"Question: '{question}'\n")
    print('Initial Vector Search Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'   Score: {match["score"]}')
        print(f'   Metadata: {match["metadata"]}\n')

print("\n--- PART 6 INITIAL RESULTS ---")
show_results(question, sorted_matches)

# 2. Prepare documents for reranking
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

# 3. Execute serverless reranking
refined_query = "acute myocardial infarction or respiratory distress symptoms" 

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

# 4. Show reranked results
def show_final_reranked_results(question, matches):
    print(f"Refined Question: '{question}'\n")
    print('Reranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'   Score: {match.score}')
        print(f'   Reranking Field: {match.document.reranking_field}\n')

print("--- PART 6 RERANKED RESULTS ---")
show_final_reranked_results(refined_query, reranked_results.data)

# 5. Clean up (Deletes the index to free resources)
pc.delete_index(name=index_name)
print(f"Index '{index_name}' safely deleted.")